# 14 CatBoost Hyperparameter Optimisierung Optuna (on 100%)

## Import

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import CatBoostPruningCallback

from catboost import CatBoostClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [3]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [4]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
full     [595212, 595212, 595212]
dtype: object

## Hilfsvariablen

In [5]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [6]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [7]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [8]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)]
    }
)

train    [476168, 476168]
val        [59522, 59522]
test       [59522, 59522]
full     [595212, 595212]
dtype: object

## Optuna Hilfe

## Suchräume

|Paramerter|Bereich|
|---|---|
|learning rate|0.01-0.3|
|depth|4-10|
|l2_leaf_reg|1-30|
|random_strength|1e3-10|
|bootstrap_type|Bayesian / Bernoulli / MVS|
|bagging_temperature|0-5|
|one_hot_max_size|2-105|
|leaf_estimation_iterations|1-10|
|auto_class_weights|none / Balanced|

In [9]:
fixed_params_plain = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Plain"
}

In [10]:
fixed_params_ordered = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Ordered"
}

In [11]:
def suchraum_params(trial):
    """Suchraum definition"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [2, 10, 18, 105]),
        "leaf_estimation_iterations": trial.suggest_int("leaf_estimation_iterations", 1, 10),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", [None, "Balanced"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    }

    if params["bootstrap_type"] == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 5.0)
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    return params

## Plain mit Calc

In [12]:
def objective_plain_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_cb, y_train, eval_set=[(x_val_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_cb)[:, 1])

In [13]:
study_plain_with_calc_full = optuna.create_study(
    study_name = "CatBoost_Plain_with_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 13:48:09,103] A new study created in RDB with name: CatBoost_Plain_with_calc_full


In [14]:
study_plain_with_calc_full.optimize(
    objective_plain_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_26220\2264105452.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 13:49:37,345] Trial 0 finished with value: 0.6307018491365225 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.24810409748678125, 'one_hot_max_size': 105, 'leaf_estimation_iterations': 7, 'auto_class_weights': None, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.9091248360355031}. Best is trial 0 with value: 0.6307018491365225.
C:\Users\linus\AppData\Local\Temp\ipykernel_26220\2264105452.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 13:53:52,364] Trial 1 finished with value: 0.6333600771

In [15]:
pd.DataFrame({
    "best auc_val": study_plain_with_calc_full.best_value,
    "best gini": 2 * study_plain_with_calc_full.best_value - 1,
    "trials": len(study_plain_with_calc_full.trials),
    "pruned": len([t for t in study_plain_with_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_with_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_plain_with_calc_full.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.633815,0.267629,50,17,0,0.063388
depth,0.633815,0.267629,50,17,0,5
l2_leaf_reg,0.633815,0.267629,50,17,0,2.430517
random_strength,0.633815,0.267629,50,17,0,0.001189
one_hot_max_size,0.633815,0.267629,50,17,0,10
leaf_estimation_iterations,0.633815,0.267629,50,17,0,3
auto_class_weights,0.633815,0.267629,50,17,0,Balanced
bootstrap_type,0.633815,0.267629,50,17,0,Bernoulli
subsample,0.633815,0.267629,50,17,0,0.565753


## Plain ohne Calc

In [16]:
def objective_plain_no_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_no_calc_cb, y_train, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [17]:
study_plain_without_calc_full = optuna.create_study(
    study_name = "CatBoost_Plain_without_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 15:06:02,132] A new study created in RDB with name: CatBoost_Plain_without_calc_full


In [18]:
study_plain_without_calc_full.optimize(
    objective_plain_no_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_26220\568654475.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 15:07:39,202] Trial 0 finished with value: 0.6353305486248135 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.24810409748678125, 'one_hot_max_size': 105, 'leaf_estimation_iterations': 7, 'auto_class_weights': None, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.9091248360355031}. Best is trial 0 with value: 0.6353305486248135.
C:\Users\linus\AppData\Local\Temp\ipykernel_26220\568654475.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 15:10:18,185] Trial 1 finished with value: 0.632708301665

In [19]:
pd.Series({
    "best auc_val": study_plain_without_calc_full.best_value,
    "best gini": 2 * study_plain_without_calc_full.best_value - 1,
    "trials": len(study_plain_without_calc_full.trials),
    "pruned": len([t for t in study_plain_without_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_without_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_plain_without_calc_full.best_params
})

best auc_val                                             0.636402
best gini                                                0.272804
trials                                                         50
pruned                                                         29
fail                                                            0
best params     {'learning_rate': 0.05445057683691509, 'depth'...
dtype: object

In [20]:
study_plain_without_calc_full.best_params

{'learning_rate': 0.05445057683691509,
 'depth': 8,
 'l2_leaf_reg': 7.076480471311773,
 'random_strength': 0.8387744815011882,
 'one_hot_max_size': 18,
 'leaf_estimation_iterations': 5,
 'auto_class_weights': None,
 'bootstrap_type': 'Bayesian',
 'bagging_temperature': 1.8158099900059381}

## Ordered ohne Calc

In [21]:
def objective_ordered_no_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_ordered,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_no_calc_cb, y_train, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [22]:
study_ordered_without_calc_full = optuna.create_study(
    study_name = "CatBoost_Ordered_without_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

[I 2026-08-13 16:27:44,274] A new study created in RDB with name: CatBoost_Ordered_without_calc_full


In [23]:
study_ordered_without_calc_full.optimize(
    objective_ordered_no_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

C:\Users\linus\AppData\Local\Temp\ipykernel_26220\1778171121.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 16:39:06,758] Trial 0 finished with value: 0.6317207829663305 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 0.24810409748678125, 'one_hot_max_size': 105, 'leaf_estimation_iterations': 7, 'auto_class_weights': None, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.9091248360355031}. Best is trial 0 with value: 0.6317207829663305.
C:\Users\linus\AppData\Local\Temp\ipykernel_26220\1778171121.py:6: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "AUC")
[I 2026-08-13 16:40:58,411] Trial 1 finished with value: 0.6296060656

In [24]:
pd.Series({
    "best auc_val": study_ordered_without_calc_full.best_value,
    "best gini": 2 * study_ordered_without_calc_full.best_value - 1,
    "trials": len(study_ordered_without_calc_full.trials),
    "pruned": len([t for t in study_ordered_without_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_ordered_without_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_ordered_without_calc_full.best_params
})

best auc_val                                             0.635504
best gini                                                0.271008
trials                                                         50
pruned                                                         19
fail                                                            0
best params     {'learning_rate': 0.10624277494598432, 'depth'...
dtype: object

In [25]:
study_ordered_without_calc_full.best_params

{'learning_rate': 0.10624277494598432,
 'depth': 7,
 'l2_leaf_reg': 18.039747295507677,
 'random_strength': 0.005187373722068937,
 'one_hot_max_size': 105,
 'leaf_estimation_iterations': 1,
 'auto_class_weights': None,
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.8119454704358904}

## Load Study

In [27]:
study_plain_with_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_with_calc_full",
    storage = "sqlite:///catboost_opti.db"
)


In [28]:
study_plain_without_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_without_calc_full",
    storage = "sqlite:///catboost_opti.db"
)

In [29]:
study_ordered_without_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Ordered_without_calc_full",
    storage = "sqlite:///catboost_opti.db"
)

## Best Params again

In [30]:
study_plain_with_calc_full_loaded.best_params

{'learning_rate': 0.06338750302989393,
 'depth': 5,
 'l2_leaf_reg': 2.430516929648288,
 'random_strength': 0.0011888525233311397,
 'one_hot_max_size': 10,
 'leaf_estimation_iterations': 3,
 'auto_class_weights': 'Balanced',
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.5657526432159653}

In [31]:
study_plain_without_calc_full_loaded.best_params

{'learning_rate': 0.05445057683691509,
 'depth': 8,
 'l2_leaf_reg': 7.076480471311773,
 'random_strength': 0.8387744815011882,
 'one_hot_max_size': 18,
 'leaf_estimation_iterations': 5,
 'auto_class_weights': None,
 'bootstrap_type': 'Bayesian',
 'bagging_temperature': 1.8158099900059381}

In [32]:
study_ordered_without_calc_full_loaded.best_params

{'learning_rate': 0.10624277494598432,
 'depth': 7,
 'l2_leaf_reg': 18.039747295507677,
 'random_strength': 0.005187373722068937,
 'one_hot_max_size': 105,
 'leaf_estimation_iterations': 1,
 'auto_class_weights': None,
 'bootstrap_type': 'Bernoulli',
 'subsample': 0.8119454704358904}

## 100% Train

In [33]:
results = []
train_times = {}

In [34]:
name = "L_cb_opt_01_full"

L_cb_opt_01_full = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_with_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_01_full.fit(
    x_train_cb, y_train, 
    eval_set=(x_val_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_01_full, x_train_cb, y_train, x_val_cb, y_val, train_times[name], best_iter=L_cb_opt_01_full.get_best_iteration())
)

0:	test: 0.5920158	best: 0.5920158 (0)	total: 117ms	remaining: 9m 44s
250:	test: 0.6296309	best: 0.6298320 (236)	total: 29.3s	remaining: 9m 15s
500:	test: 0.6333300	best: 0.6333876 (474)	total: 1m	remaining: 9m 6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.63381464
bestIteration = 623

Shrink model to first 624 iterations.


In [35]:
name = "L_cb_opt_02_full"

L_cb_opt_02_full = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_without_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_02_full.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_02_full, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_02_full.get_best_iteration())
)

0:	test: 0.5655829	best: 0.5655829 (0)	total: 177ms	remaining: 14m 43s
250:	test: 0.6320901	best: 0.6320901 (250)	total: 43.9s	remaining: 13m 49s
500:	test: 0.6358414	best: 0.6360036 (494)	total: 1m 27s	remaining: 13m 8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6364019991
bestIteration = 558

Shrink model to first 559 iterations.


In [36]:
name = "L_cb_opt_03_full"

L_cb_opt_03_full = CatBoostClassifier(
    **fixed_params_ordered,
    **study_ordered_without_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_03_full.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_03_full, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_03_full.get_best_iteration())
)

0:	test: 0.5575105	best: 0.5575105 (0)	total: 92.3ms	remaining: 7m 41s
250:	test: 0.6324803	best: 0.6324939 (249)	total: 34.7s	remaining: 10m 56s
500:	test: 0.6345583	best: 0.6345583 (500)	total: 1m 9s	remaining: 10m 25s
750:	test: 0.6352451	best: 0.6353866 (732)	total: 1m 44s	remaining: 9m 51s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6355038302
bestIteration = 789

Shrink model to first 790 iterations.


In [37]:
pd.DataFrame(results)

,model_idx,auc_train,auc_val,gini,delta_auc,best_iter,trainingszeit
0,L_cb_opt_01_full,0.696718,0.633815,0.267629,0.062904,623,88.705495
1,L_cb_opt_02_full,0.692775,0.636402,0.272804,0.056373,558,116.162328
2,L_cb_opt_03_full,0.685571,0.635504,0.271008,0.050067,789,124.465556


## Save Models

In [38]:
Modelle = [
    (L_cb_opt_01_full, "L_cb_opt_01_full"),
    (L_cb_opt_02_full, "L_cb_opt_02_full"),
    (L_cb_opt_03_full, "L_cb_opt_03_full")
]

for modell, name in Modelle:
    modell.save_model(f"{name}.cbm")

## Notizen
- Die Learning Curves zeigten kein definitives Plateau, also optimiere ich hier nochmal anhand des vollen train sets
- ich setze die zeitbegrenzung mal auf 4 stunden zur sicherheit


